In [ ]:
import time

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import k_hop_subgraph
from torch_geometric.nn import SAGEConv, global_mean_pool
from sklearn.metrics import roc_auc_score

def drnl_node_labeling(adj, src, dst, simple_version=False):
    # Double Radius Node Labeling (DRNL).
    if not simple_version:
        src, dst = (dst, src) if src > dst else (src, dst)

        idx = list(range(src)) + list(range(src + 1, adj.shape[0]))
        adj_wo_src = adj[idx, :][:, idx]

        idx = list(range(dst)) + list(range(dst + 1, adj.shape[0]))
        adj_wo_dst = adj[idx, :][:, idx]

        dist2src = shortest_path(adj_wo_dst, directed=False, unweighted=True, indices=src)
        dist2src = np.insert(dist2src, dst, 0, axis=0)
        dist2src = torch.from_numpy(dist2src)

        dist2dst = shortest_path(adj_wo_src, directed=False, unweighted=True, indices=dst-1)
        dist2dst = np.insert(dist2dst, src, 0, axis=0)
        dist2dst = torch.from_numpy(dist2dst)

        dist = dist2src + dist2dst
        dist_over_2, dist_mod_2 = dist // 2, dist % 2

        z = 1 + torch.min(dist2src, dist2dst)
        z += dist_over_2 * (dist_over_2 + dist_mod_2 - 1)
        z[src] = 1.
        z[dst] = 1.
        z[torch.isnan(z)] = 0.
    else:
        # 간단한 구조 특징 추가: 소스/타깃 노드엔 1.0, 주변 노드엔 0.0 부여 (DRNL의 간소화 버전)
        z = torch.zeros((adj.shape[0], 1), dtype=torch.float)
        z[src] = 1.0
        z[dst] = 1.0

    return z.to(torch.long)

class SEALSubgraphDataset(torch.utils.data.Dataset):
    def __init__(self, full_data, link_index, link_labels, num_hops=2):
        '''
        full_data: 원본 데이터
        link_index: 링크 쌍 [2, num_edges]
        link_labels: 1 = 링크 존재 0 = 링크  존재 x
        '''
        self.x = full_data.x
        self.edge_index = full_data.edge_index
        self.links = link_index.t().tolist()
        self.labels = link_labels.tolist()
        self.num_hops = num_hops

    def __len__(self):
        return len(self.links)
    
    def __getitem__(self, idx):
        src, dst = self.links[idx]
        y = self.labels[idx]

        # 두 노드 주변의 k-hop 서브 그래프 추출
        subnodes, subedges, mapping, edge_mask = k_hop_subgraph(
            node_idx=[src, dst],
            num_hops=self.num_hops,
            edge_index=self.edge_index,
            relabel_nodes=True
        )

        # 서브그래프에 속한 노드들의 원래 Feature 추출
        sub_x = self.x[subnodes]

        num_nodes = len(subnodes)
        row = subedges[0]
        col = subedges[1]
        weight = np.ones(subedges.shape[1], dtype=int)
        adj = csr_matrix((weight, (row, col)), shape=(num_nodes, num_nodes))

        # 원래의 소스(src), 타깃( dst) 노드가 서브 그래프 안에서 몇 번 인덱스로 바뀌었는지 확인
        # 이 mapping 정보를 통해 DRNL 등 구조적 라벨링을 추가할 수 있음.
        src_new, dst_new = mapping[0].item(), mapping[1].item()

        # 간단한 구조 특징 추가: 소스/타깃 노드엔 1.0, 주변 노드엔 0.0 부여 (DRNL의 간소화 버전)
        structural_feat = torch.zeros((sub_x.size(0), 1), dtype=torch.float)
        structural_feat[src_new] = 1.0
        structural_feat[dst_new] = 1.0
        print(structural_feat.shape, 'struc')
        z = drnl_node_labeling(adj, src_new, dst_new, simple_version=False)
        z = z.reshape(-1, 1)
        print(z.shape, 'z')

        final_x = torch.cat([sub_x, structural_feat], dim=-1)

        # PyG의 Data 객체와 유사하게
        return {
            'x': final_x,
            'edge_index': subedges,
            'y': torch.tensor(y, dtype=torch.float)
        }
    
def collate_fn(batch):
    # 여러 서브 그래프를 하나의 그래프로 합침.
    num_nodes_cum = 0
    batch_x = []
    batch_edge_index = []
    batch_y = []
    batch_index = []
    
    for i, data in enumerate(batch):
        x, edge_index, y = data['x'], data['edge_index'], data['y']
        num_nodes = x.size(0)

        batch_x.append(x)
        batch_edge_index.append(edge_index + num_nodes_cum)
        batch_y.append(y)
        # 어떤 노드가 몇 번째 서브그래프(배치)에 속하는 기록
        batch_index.append(torch.full((num_nodes,), i, dtype=torch.long))

        num_nodes_cum += num_nodes

    return {
        'x': torch.cat(batch_x, dim=0),
        'edge_index': torch.cat(batch_edge_index, dim=1),
        'y': torch.stack(batch_y, dim=0), 
        'batch': torch.cat(batch_index, dim=0)
    }


dataset = Planetoid(root='./data/Cora', name="Cora")
cora_data = dataset[0]

transform = RandomLinkSplit(num_val=0.1, num_test=0.1, is_undirected=True, add_negative_train_samples=True)
train_data, val_data, test_data = transform(cora_data)

train_dataset = SEALSubgraphDataset(train_data, train_data.edge_label_index, train_data.edge_label)
val_dataset = SEALSubgraphDataset(val_data, val_data.edge_label_index, val_data.edge_label)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

class SEALClassifier(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin1 = torch.nn.Linear(hidden_channels, 64)
        self.lin2 = torch.nn.Linear(64, 1)

    def forward(self, x, edge_index, batch):
        # 1. 서브 그래프 내부 노드 전파
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))

        # 2. Graph-level Pooling (각 서브그래프별로 하나의 벡터 추출)
        h = global_mean_pool(h, batch)

        # 3. 최종 분류 
        h = F.relu(self.lin1(h))
        h = F.dropout(h, p=0.5, training=self.training)
        return torch.sigmoid(self.lin2(h)).squeeze(-1)
    

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SEALClassifier(in_channels=dataset.num_features + 1, hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCELoss()

def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch['x'].to(device), batch['edge_index'].to(device), batch['batch'].to(device))
        loss = criterion(out, batch['y'].to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds, targets = [], []
    for batch in loader:
        out = model(batch['x'].to(device), batch['edge_index'].to(device), batch['batch'].to(device))
        preds.extend(out.cpu().tolist())
        targets.extend(batch['y'].tolist())
    return  roc_auc_score(targets, preds)

print('------학습시작-------')
start_time = time.time()
for epoch in range(1, 11):
    loss = train()
    val_auc = evaluate(val_loader)
    print(f"Epoch: {epoch:02d}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}")

print(f"--------학습 완료--------, 총 걸린시간(s): {time.time() - start_time}")



------학습시작-------
torch.Size([153, 1]) struc
torch.Size([153]) z
torch.Size([65, 1]) struc
torch.Size([65]) z
torch.Size([15, 1]) struc
torch.Size([15]) z
torch.Size([100, 1]) struc
torch.Size([100]) z
torch.Size([37, 1]) struc
torch.Size([37]) z
torch.Size([139, 1]) struc
torch.Size([139]) z
torch.Size([40, 1]) struc
torch.Size([40]) z
torch.Size([30, 1]) struc
torch.Size([30]) z
torch.Size([165, 1]) struc
torch.Size([165]) z
torch.Size([143, 1]) struc
torch.Size([143]) z
torch.Size([101, 1]) struc
torch.Size([101]) z
torch.Size([62, 1]) struc
torch.Size([62]) z
torch.Size([19, 1]) struc
torch.Size([19]) z
torch.Size([17, 1]) struc
torch.Size([17]) z
torch.Size([36, 1]) struc
torch.Size([36]) z
torch.Size([27, 1]) struc
torch.Size([27]) z
torch.Size([127, 1]) struc
torch.Size([127]) z
torch.Size([58, 1]) struc
torch.Size([58]) z
torch.Size([57, 1]) struc
torch.Size([57]) z
torch.Size([27, 1]) struc
torch.Size([27]) z
torch.Size([43, 1]) struc
torch.Size([43]) z
torch.Size([29, 1]) str

In [ ]:
from scipy.sparse.csgraph import shortest_path
import numpy as np




In [1]:
import torch
from scipy.sparse import csr_matrix
edge_list = np.array([[0, 1, 2, 3, 4, 5],
                     [ 2, 2, 4, 4, 6, 6]])

num_nodes = 7
row = edge_list[0]
col = edge_list[1]

data = np.ones(edge_list.shape[1], dtype=int)
adj = csr_matrix((data, (row, col)), shape=(num_nodes, num_nodes))

src, dst = 0, 1

z = drnl_node_labeling(adj, src, dst)
print(z)

NameError: name 'np' is not defined

In [ ]:
adj.shape[0]

7